# Dataset Initialization

### Constants

In [1]:
import json
import torch
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, utils
from datasets import load_dataset

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"

utils.logging.set_verbosity_error()  # Suppress standard warnings

models = [
        'meta-llama/Llama-3.2-1B', # Llama-3.2-1B (16 layers 32 heads)
        'meta-llama/Meta-Llama-3-8B', # Meta-Llama-3-8B
        'meta-llama/Meta-Llama-3-70B', # Meta-Llama-3-70B
        'meta-llama/Meta-Llama-3-8B-Instruct', # Meta-Llama-3-8B-Instruct (the only model TRAIL tests with)
        'EleutherAI/gpt-j-6b', # GPT-J 
        'meta-llama/Llama-2-13b-chat-hf', # Llama-2, fine-tuned
        'EleutherAI/gpt-neox-20b', # GPT-NeoX
        ]

# datasets = {
#     1: './data/dataset_alpaca.json',
#     2: './data/datasetSimplified_alpaca.json',
#     3: './data/dataset_lmsys-chat-1m.json',
#     4: 'yahma/alpaca-cleaned'
# }

CACHE_DIR = "～/.cache/huggingface/datasets"
DS_NAME = "yahma/alpaca-cleaned"

# Parameters
max_new_tokens = 300
temperature = 1  # Lower temperature for more deterministic output
top_k = 1  # Increase top_k for more diverse candidates
repetition_penalty = 1.3  # Increase repetition penalty to reduce repetition

### Dataset Loading

In [ ]:
ds = load_dataset(DS_NAME)
dataset = ds['train']

prompt_template = "{instruction}\n\n{input}"  # template for the prompt
prompts = []

for inst, inp in zip(dataset["instruction"], dataset["input"]):
    if inp.strip() == "":  # no input
        prompt = inst
    else:
        prompt = prompt_template.format(instruction=inst, input=inp)
    prompts.append(prompt)

# structure the data
data = {
    "qa_pairs": [
        {"prompt": prompt, "response": output}
        for prompt, output in zip(prompts, dataset["output"])
    ]
}

# print the output
# print(json.dumps(data, indent=2))

### Model Loading

In [ ]:
print(
    "Choose the model to test:\n"
    " 1. Llama-3.2-1B\n"
    " 2. Meta-Llama-3-8B\n"
    " 3. Meta-Llama-3-70B\n"
    " 4. Meta-Llama-3-8B-Instruct\n"
    " 5. GPT-J\n"
    " 6. Llama-2\n"
    " 7. GPT-NeoX"
)
model_choice = int(input("Enter the number corresponding to the model: "))
if model_choice < 1 or model_choice > len(models):
    raise ValueError("Invalid model choice. Please enter a number between 1 and 7.")
model_name = models[model_choice - 1]

tokenizer = AutoTokenizer.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=CACHE_DIR, device_map="auto", attn_implementation="eager")

eos_token_id = tokenizer.eos_token_id
eos_flag=0